# 1. Establish a connection between Python and the Sakila database.

In [1]:
!pip install sqlalchemy pymysql pandas


In [2]:
import pandas as pd
from sqlalchemy import create_engine

In [6]:
user = "root"
password = "**********"
host = "localhost"
database = "sakila"

engine = create_engine(
    f"mysql+pymysql://{user}:{password}@{host}/{database}"
)

In [7]:
pd.read_sql("SELECT VERSION();", engine)

,VERSION()
0,8.0.46


# 2. Write a Python function called rentals_month that retrieves rental data for a given month and 
# year (passed as parameters) from the Sakila database as a Pandas DataFrame. 
# The function should take in three parameters:

# engine: an object representing the database connection engine to be used to establish a connection to the Sakila database.
# month: an integer representing the month for which rental data is to be retrieved.
# year: an integer representing the year for which rental data is to be retrieved.
# The function should execute a SQL query to retrieve the rental data for the specified month and year from the rental table 
# in the Sakila database, and return it as a pandas DataFrame.


In [19]:
from sqlalchemy import text

def rentals_month(engine, month, year):

    query = text("""
        SELECT *
        FROM rental
        WHERE MONTH(rental_date) = :month
          AND YEAR(rental_date) = :year
    """)

    rentals_df = pd.read_sql(
        query,
        engine,
        params={
            "month": month,
            "year": year
        }
    )

    return rentals_df

In [20]:
# test 

may_rentals = rentals_month(engine, 5, 2005)

may_rentals.head()

,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1,2005-05-24 22:53:30,367,130,2005-05-26 22:04:30,1,2006-02-15 21:30:53
1,2,2005-05-24 22:54:33,1525,459,2005-05-28 19:40:33,1,2006-02-15 21:30:53
2,3,2005-05-24 23:03:39,1711,408,2005-06-01 22:12:39,1,2006-02-15 21:30:53
3,4,2005-05-24 23:04:41,2452,333,2005-06-03 01:43:41,2,2006-02-15 21:30:53
4,5,2005-05-24 23:05:21,2079,222,2005-06-02 04:33:21,1,2006-02-15 21:30:53


In [ ]:
may_rentals = rentals_month(engine, 5, 2005)
june_rentals = rentals_month(engine, 6, 2005)

In [17]:
may_rentals.head()

,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1,2005-05-24 22:53:30,367,130,2005-05-26 22:04:30,1,2006-02-15 21:30:53
1,2,2005-05-24 22:54:33,1525,459,2005-05-28 19:40:33,1,2006-02-15 21:30:53
2,3,2005-05-24 23:03:39,1711,408,2005-06-01 22:12:39,1,2006-02-15 21:30:53
3,4,2005-05-24 23:04:41,2452,333,2005-06-03 01:43:41,2,2006-02-15 21:30:53
4,5,2005-05-24 23:05:21,2079,222,2005-06-02 04:33:21,1,2006-02-15 21:30:53


In [21]:
june_rentals.head()

,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1158,2005-06-14 22:53:33,1632,416,2005-06-18 21:37:33,2,2006-02-15 21:30:53
1,1159,2005-06-14 22:55:13,4395,516,2005-06-17 02:11:13,1,2006-02-15 21:30:53
2,1160,2005-06-14 23:00:34,2795,239,2005-06-18 01:58:34,2,2006-02-15 21:30:53
3,1161,2005-06-14 23:07:08,1690,285,2005-06-21 17:12:08,1,2006-02-15 21:30:53
4,1162,2005-06-14 23:09:38,987,310,2005-06-23 22:00:38,1,2006-02-15 21:30:53


# 3. Develop a Python function called rental_count_month that takes the DataFrame provided by rentals_month 
# as input along with the month and year and returns a new DataFrame containing the number of rentals made by each customer_id 
# during the selected month and year.

# The function should also include the month and year as parameters and use them to name the new column according to the month and year, 
# for example, if the input month is 05 and the year is 2005, the column name should be "rentals_05_2005".

# Hint: Consider making use of pandas groupby()


In [16]:
def rental_count_month(df, month, year):
    column_name = f"rentals_{month:02d}_{year}"

    rental_counts = (
        df.groupby("customer_id")
          .size()
          .reset_index(name=column_name)
    )

    return rental_counts


may_counts = rental_count_month(may_rentals, 5, 2005)
june_counts = rental_count_month(june_rentals, 6, 2005)

In [25]:
may_counts.head()

,customer_id,may_rentals
0,1,2
1,2,1
2,3,2
3,5,3
4,6,3


# 4. Create a Python function called compare_rentals that takes two DataFrames as input 
# containing the number of rentals made by each customer in different months and years. 
# The function should return a combined DataFrame with a new 'difference' column, which is 
# the difference between the number of rentals in the two months.

In [26]:
def compare_rentals(df1, df2):

    comparison = df1.merge(
        df2,
        on="customer_id",
        how="inner"
    )

    rental_columns = comparison.columns[1:]

    comparison["difference"] = (
        comparison[rental_columns[1]] -
        comparison[rental_columns[0]]
    )

    return comparison

In [27]:
comparison = compare_rentals(may_counts, june_counts)
comparison.head()

,customer_id,may_rentals,june_rentals,difference
0,1,2,7,5
1,2,1,1,0
2,3,2,4,2
3,5,3,5,2
4,6,3,4,1
